<a href="https://colab.research.google.com/github/NBall65097/Fantasy-Story-Weaver/blob/main/Fantasy_Story_Weaver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers==0.0.28.post2 trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-uyeuhf3a/unsloth_f4ddb57a8f864481bcb51f0c803ca9b0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-uyeuhf3a/unsloth_f4ddb57a8f864481bcb51f0c803ca9b0
  Resolved https://github.com/unslothai/unsloth.git to commit 9a551831ef2eff33f947cc1a7e1b259f9913950b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 23.7 MB/s eta 0:00:0

In [ ]:
from huggingface_hub import login; login() # Connect to HuggingFace account

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import datasets # datasets function for importing data

In [ ]:
filepath = "/content/data/FSTDataset.jsonl"
data = datasets.load_dataset("json",data_files=filepath,split='train')
data.push_to_hub('NBall65097/fantasy-storyweaver-data') # Save dataset to HuggingFace

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 34.9kB / 34.9kB            

README.md:   0%|          | 0.00/339 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/NBall65097/fantasy-storyweaver-data/commit/2c3bd32357f307c3b71105a522949157f45e2f55', commit_message='Upload dataset', commit_description='', oid='2c3bd32357f307c3b71105a522949157f45e2f55', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/NBall65097/fantasy-storyweaver-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='NBall65097/fantasy-storyweaver-data'), pr_revision=None, pr_num=None)

In [ ]:
data = data.train_test_split(test_size=0.1, train_size=0.9)
traindata, testdata = data["train"], data["test"]

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import unsloth
from unsloth import FastLanguageModel

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    dtype=None,  # auto
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
dataset = load_dataset('NBall65097/fantasy-storyweaver-data',split="train")

In [ ]:
from unsloth.chat_templates import apply_chat_template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="phi-3")

def format_phi35_data(examples):
  formatted_texts = []
  for msg_list in examples["messages"]:
    text = tokenizer.apply_chat_template(
        msg_list,
        tokenize=False,
        add_generation_prompt=False,
    )
    formatted_texts.append(text)
  return {"text":formatted_texts}

In [ ]:
dataset = dataset.map(
    format_phi35_data,
    batched=True,
    batch_size=1000,          # adjust based on RAM
    num_proc=2,               # or 4 if you have enough CPU cores
    desc="Formatting Phi-3.5 chat examples",
)

Formatting Phi-3.5 chat examples (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=formatting_func,
    dataset_text_field="text",  # or use formatting_func for chat template
    max_seq_length=2048,
    packing=True,  # packs multiple examples → huge speed win
    args=SFTConfig(
        output_dir="/content/output",
        num_train_epochs=2,
        learning_rate=2e-4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        fp16=True,
        gradient_checkpointing=True,
        optim="adamw_8bit",
        logging_steps=10,
        save_strategy="epoch",
        save_steps=200,
        report_to="none",
    )  # TrainingArguments: 1–3 epochs, lr=2e-4, batch=4–8 (gradient accum=4), warmup, etc.
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32009}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 2 | Total steps = 8
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Step,Training Loss


TrainOutput(global_step=8, training_loss=2.194103479385376, metrics={'train_runtime': 108.8345, 'train_samples_per_second': 0.919, 'train_steps_per_second': 0.074, 'total_flos': 1062472475750400.0, 'train_loss': 2.194103479385376, 'epoch': 2.0})

In [ ]:
model.save_pretrained("fantasy-weaver-lora")          # saves adapter only (~50–200 MB)
tokenizer.save_pretrained("fantasy-weaver-lora")

('fantasy-weaver-lora/tokenizer_config.json',
 'fantasy-weaver-lora/chat_template.jinja',
 'fantasy-weaver-lora/tokenizer.json')